In [1]:
import pandas as pd
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
notes_val = pd.read_csv("notes_val_predictions_task5.csv")
ecg_val = pd.read_csv("ecg_val_predictions_task5.csv")
structured_val = pd.read_csv("structured_val_predictions_task5.csv")

display(notes_val.head())
display(ecg_val.head())
display(structured_val.head())

print(len(notes_val))
print(len(ecg_val))
print(len(structured_val))

,sample_id,y_true,pred_prob_notes
0,0,1,0.993320
1,1,1,0.909586
2,2,1,0.837753
3,3,1,0.991554
4,4,1,0.943348


,sample_id,y_true,pred_prob_ecg
0,0,1,0.802869
1,1,1,0.766404
2,2,1,0.638059
3,3,1,0.936038
4,4,1,0.612819


,sample_id,y_true,pred_prob_structured
0,0,1,0.997023
1,1,1,0.998733
2,2,1,0.999352
3,3,1,0.996012
4,4,1,0.998882


18986
18986
18986


In [3]:
val_df = notes_val.merge(
    ecg_val[["sample_id", "pred_prob_ecg"]],
    on="sample_id",
    how="inner"
).merge(
    structured_val[["sample_id", "pred_prob_structured"]],
    on="sample_id",
    how="inner"
)

val_df

,sample_id,y_true,pred_prob_notes,pred_prob_ecg,pred_prob_structured
0,0,1,0.993320,0.802869,0.997023
1,1,1,0.909586,0.766404,0.998733
2,2,1,0.837753,0.638059,0.999352
3,3,1,0.991554,0.936038,0.996012
4,4,1,0.943348,0.612819,0.998882
...,...,...,...,...,...
18981,18981,1,0.913984,0.850202,0.998013
18982,18982,1,0.970937,0.821923,0.998718
18983,18983,1,0.341082,0.601516,0.999071
18984,18984,0,0.266262,0.380964,0.008124


In [4]:
notes_test = pd.read_csv("notes_test_predictions_task5.csv")
ecg_test = pd.read_csv("ecg_test_predictions_task5.csv")
structured_test = pd.read_csv("structured_test_predictions_task5.csv")

display(notes_test.head())
display(ecg_test.head())
display(structured_test.head())

print(len(notes_test))
print(len(ecg_test))
print(len(structured_test))

,sample_id,y_true,pred_prob_notes
0,0,1,0.929280
1,1,0,0.384999
2,2,0,0.073863
3,3,0,0.760178
4,4,0,0.157850


,sample_id,y_true,pred_prob_ecg
0,0,1,0.503109
1,1,0,0.814529
2,2,0,0.596409
3,3,0,0.674331
4,4,0,0.360603


,sample_id,y_true,pred_prob_structured
0,0,1,0.987616
1,1,0,0.011343
2,2,0,0.012877
3,3,0,0.004147
4,4,0,0.003101


18987
18987
18987


In [5]:
test_df = notes_test.merge(
    ecg_test[["sample_id", "pred_prob_ecg"]],
    on="sample_id",
    how="inner"
).merge(
    structured_test[["sample_id", "pred_prob_structured"]],
    on="sample_id",
    how="inner"
)
test_df

,sample_id,y_true,pred_prob_notes,pred_prob_ecg,pred_prob_structured
0,0,1,0.929280,0.503109,0.987616
1,1,0,0.384999,0.814529,0.011343
2,2,0,0.073863,0.596409,0.012877
3,3,0,0.760178,0.674331,0.004147
4,4,0,0.157850,0.360603,0.003101
...,...,...,...,...,...
18982,18982,1,0.825203,0.815805,0.998968
18983,18983,1,0.823896,0.871649,0.998984
18984,18984,1,0.991859,0.635257,0.998859
18985,18985,0,0.875095,0.916166,0.003342


In [6]:
# Feature engineering
def add_meta_features(df):
    eps = 1e-6
    
    # base probs
    s = df["pred_prob_structured"].astype(float)
    n = df["pred_prob_notes"].astype(float)
    e = df["pred_prob_ecg"].astype(float)
    
    # interaction terms
    df["notes_x_ecg"] = n * e
    df["notes_x_struct"] = n * s
    df["ecg_x_struct"] = e * s
    df["notes_x_ecg_x_struct"] = n * e * s
    
    # pairwise differences
    df["struct_minus_notes"] = s - n
    df["struct_minus_ecg"] = s - e
    df["notes_minus_ecg"] = n - e
    
    # summary stats across modalities
    df["prob_mean"] = (s + n + e) / 3.0
    df["prob_max"] = np.maximum(np.maximum(s, n), e)
    df["prob_min"] = np.minimum(np.minimum(s, n), e)
    df["prob_range"] = df["prob_max"] - df["prob_min"]
    
    # agreement / dispersion
    df["prob_std"] = np.std(np.vstack([s, n, e]), axis=0)
    
    # logit transform can help logistic stacking
    df["logit_struct"] = np.log((s + eps) / (1 - s + eps))
    df["logit_notes"] = np.log((n + eps) / (1 - n + eps))
    df["logit_ecg"] = np.log((e + eps) / (1 - e + eps))
    
    # rank-like indicators / thresholds
    df["struct_gt_notes"] = (s > n).astype(int)
    df["struct_gt_ecg"] = (s > e).astype(int)
    df["notes_gt_ecg"] = (n > e).astype(int)
    
    # confidence-style features
    df["struct_conf"] = np.abs(s - 0.5)
    df["notes_conf"] = np.abs(n - 0.5)
    df["ecg_conf"] = np.abs(e - 0.5)
    
    return df

val_df = add_meta_features(val_df.copy())
test_df = add_meta_features(test_df.copy())

# Features
stack_features = [
    # raw probs
    "pred_prob_structured",
    "pred_prob_notes",
    "pred_prob_ecg",
    
    # interactions
    "notes_x_ecg",
    "notes_x_struct",
    "ecg_x_struct",
    "notes_x_ecg_x_struct",
    
    # differences
    "struct_minus_notes",
    "struct_minus_ecg",
    "notes_minus_ecg",
    
    # summaries
    "prob_mean",
    "prob_max",
    "prob_min",
    "prob_range",
    "prob_std",
    
    # logits
    "logit_struct",
    "logit_notes",
    "logit_ecg",
    
    # comparisons
    "struct_gt_notes",
    "struct_gt_ecg",
    "notes_gt_ecg",
    
    # confidence
    "struct_conf",
    "notes_conf",
    "ecg_conf",
]

X_val = val_df[stack_features].replace([np.inf, -np.inf], np.nan).fillna(0)
y_val = val_df["y_true"].astype(int)

X_test = test_df[stack_features].replace([np.inf, -np.inf], np.nan).fillna(0)
y_test = test_df["y_true"].astype(int)

# Logistic stacking model
meta_model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        l1_ratio=0.2,
        C=0.5,
        max_iter=5000,
        class_weight="balanced",
        random_state=42
    ))
])

meta_model.fit(X_val, y_val)

# Predict + evaluate
test_df["pred_logistic_ensemble"] = meta_model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, test_df["pred_logistic_ensemble"])
auprc = average_precision_score(y_test, test_df["pred_logistic_ensemble"])

print("Logistic multimodal test AUROC:", round(auc, 6))
print("Logistic multimodal test AUPRC:", round(auprc, 6))

# Inspect learned weights
lr = meta_model.named_steps["lr"]
coefs = pd.DataFrame({
    "feature": stack_features,
    "coef": lr.coef_[0]
}).sort_values("coef", ascending=False)

print("\nTop positive coefficients:")
print(coefs.head(10).to_string(index=False))

print("\nTop negative coefficients:")
print(coefs.tail(10).to_string(index=False))

Logistic multimodal test AUROC: 0.998368
Logistic multimodal test AUPRC: 0.999226

Top positive coefficients:
             feature     coef
        logit_struct 5.478776
         logit_notes 0.699351
         notes_x_ecg 0.436184
notes_x_ecg_x_struct 0.417806
        notes_gt_ecg 0.369986
        ecg_x_struct 0.360760
     struct_gt_notes 0.273742
           logit_ecg 0.099580
            prob_max 0.098854
            prob_std 0.063111

Top negative coefficients:
             feature      coef
  struct_minus_notes -0.007014
     notes_minus_ecg -0.090818
       pred_prob_ecg -0.116919
    struct_minus_ecg -0.168501
          notes_conf -0.178655
            ecg_conf -0.209919
pred_prob_structured -0.269911
     pred_prob_notes -0.279482
           prob_mean -0.358589
      notes_x_struct -0.400401
